In [ ]:
# xóa tất cả các file, giữ lại file Grocery1.v1i.yolov8.zip
!find /content -mindepth 1 -not -name 'Grocery1.v1i.yolov8.zip' -delete

In [ ]:
!pip install -q ultralytics

# 1. Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

import zipfile
from pathlib import Path
from collections import Counter

zip_name = next(iter(uploaded.keys()))
dataset_dir = Path("/content/bep_nha_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

# Giải nén dataset
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(dataset_dir)

# Các đuôi file thường gặp trong dataset YOLO
allowed_extensions = {
    # Ảnh
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff",

    # Nhãn YOLO
    ".txt",

    # Cấu hình dataset
    ".yaml", ".yml",

    # File dữ liệu bổ sung / metadata
    ".json", ".csv", ".xml",

    # File hệ thống hoặc hướng dẫn
    ".md", ".pdf", ".gitignore",
}

# Các file hệ thống phổ biến, không cần tính là file lạ
ignored_names = {
    ".DS_Store",
    "Thumbs.db",
    "desktop.ini",
}

all_files = [p for p in dataset_dir.rglob("*") if p.is_file()]

unknown_files = []
extension_counts = Counter()
ignored_files = []

for path in all_files:
    ext = path.suffix.lower()
    extension_counts[ext if ext else "[không có đuôi]"] += 1

    if path.name in ignored_names or "__MACOSX" in path.parts:
        ignored_files.append(path)
    elif ext not in allowed_extensions:
        unknown_files.append(path)

print(f"Tổng số file: {len(all_files)}")

print("\n=== THỐNG KÊ ĐUÔI FILE ===")
for ext, count in sorted(extension_counts.items()):
    print(f"{ext}: {count} file")

print("\n=== FILE LẠ CẦN KIỂM TRA ===")
if unknown_files:
    for path in unknown_files:
        print(f"[LẠ] {path.relative_to(dataset_dir)}")
    print(f"\nTổng file lạ: {len(unknown_files)}")
else:
    print("Không phát hiện file có đuôi ngoài danh sách cho phép.")

print("\n=== FILE HỆ THỐNG BỎ QUA ===")
if ignored_files:
    for path in ignored_files:
        print(path.relative_to(dataset_dir))
else:
    print("Không có.")

print("\n=== FILE data.yaml ===")
yaml_files = list(dataset_dir.rglob("data.yaml"))
if yaml_files:
    for path in yaml_files:
        print(path)
else:
    print("Không tìm thấy data.yaml")


In [ ]:
# upzip file neu khong dung code de upload
from pathlib import Path
import zipfile

zip_path = Path("/content/Food ingredients.v1i.yolov8.zip")
extract_path = Path("/content/bep_nha_dataset")

print("File tồn tại:", zip_path.exists())
print("ZIP hợp lệ:", zipfile.is_zipfile(zip_path))

if not zipfile.is_zipfile(zip_path):
    raise ValueError("File không phải ZIP hợp lệ. Kiểm tra lại file tải lên.")

extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Giải nén thành công!")
print("Các file/thư mục:", [p.name for p in extract_path.iterdir()])

In [ ]:
!ls -lh /content

In [ ]:

from pathlib import Path
from collections import Counter

dataset_dir = Path("/content/bep_nha_dataset")

print("=== CẤU TRÚC THƯ MỤC ===")

for path in sorted(dataset_dir.rglob("*")):
    relative = path.relative_to(dataset_dir)

    if path.is_dir():
        # Chỉ in các thư mục, giới hạn độ sâu để tránh quá dài
        if len(relative.parts) <= 4:
            print(f"[DIR]  {relative}/")
    elif path.suffix.lower() == ".txt":
        print(f"[TXT]  {relative}")
    elif path.name.lower() in {"data.yaml", "data.yml"}:
        print(f"[YAML] {relative}")

print("\n=== THỐNG KÊ ẢNH THEO THƯ MỤC ===")

image_counts = Counter(
    str(path.parent.relative_to(dataset_dir))
    for path in dataset_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)

for folder, count in sorted(image_counts.items()):
    print(f"{folder}: {count} ảnh")


In [ ]:
import yaml

if not yaml_files:
    raise FileNotFoundError(
        "Không tìm thấy data.yaml. Kiểm tra lại file ZIP."
    )

DATA_YAML = yaml_files[0]

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("Dataset:", DATA_YAML)
print("Classes:", config.get("names"))
print("Train:", config.get("train"))
print("Validation:", config.get("val"))

# 2. Format lại label

## a. Đọc data.yaml và xem các label sẽ được gộp

In [ ]:
from pathlib import Path
from collections import defaultdict
import yaml

dataset_dir = Path("/content/bep_nha_dataset")

# Tìm data.yaml trong thư mục đã giải nén
yaml_files = list(dataset_dir.rglob("data.yaml"))

if len(yaml_files) != 1:
    raise ValueError(
        f"Cần xác định đúng một data.yaml, tìm thấy {len(yaml_files)} file:\n"
        + "\n".join(str(p) for p in yaml_files)
    )

data_yaml = yaml_files[0]

with open(data_yaml, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

old_names = data["names"]
if isinstance(old_names, dict):
    old_names = [
        old_names[i] for i in sorted(old_names, key=lambda x: int(x))
    ]

# Chuẩn hóa chữ hoa/thường và khoảng trắng
names_lower = [str(name).strip().lower() for name in old_names]
available_names = set(names_lower)


def singular_candidates(name):
    """Đề xuất dạng số ít cho từ cuối của tên class."""
    parts = name.rsplit(" ", 1)
    prefix = parts[0] + " " if len(parts) == 2 else ""
    word = parts[-1]
    candidates = []

    if word.endswith("ies") and len(word) > 3:
        candidates.append(prefix + word[:-3] + "y")

    if word.endswith("es") and len(word) > 2:
        candidates.append(prefix + word[:-2])

    if word.endswith("s") and not word.endswith("ss") and len(word) > 1:
        candidates.append(prefix + word[:-1])

    return candidates


def normalize_class(name):
    name = str(name).strip().lower()

    # Chỉ gộp số nhiều khi dạng số ít có trong danh sách class.
    for candidate in singular_candidates(name):
        if candidate in available_names:
            return candidate

    return name


normalized_names = [normalize_class(n) for n in old_names]

groups = defaultdict(list)
for old_id, old_name in enumerate(old_names):
    groups[normalized_names[old_id]].append((old_id, old_name))

print("Dataset:", dataset_dir)
print("YAML:", data_yaml)
print("Số class ban đầu:", len(old_names))
print("Số class sau chuẩn hóa:", len(groups))

print("\n=== CÁC NHÓM CÓ NHIỀU LABEL ===")
for new_name, items in sorted(groups.items()):
    if len(items) > 1:
        print(f"{new_name}: " + ", ".join(
            f"{name} (ID {idx})" for idx, name in items
        ))

## b. Tạo dataset mới và cập nhật ID nhãn YOLO

In [ ]:
import shutil

output_dir = Path("/content/bep_nha_dataset_normalized")

if output_dir.exists():
    raise FileExistsError(
        f"Thư mục đầu ra đã tồn tại: {output_dir}. "
        "Hãy đổi tên hoặc xóa bản cũ trước khi chạy lại."
    )

# Tạo bản sao, không sửa dataset gốc
shutil.copytree(dataset_dir, output_dir)

# Ánh xạ class ID cũ sang ID mới
new_names = list(dict.fromkeys(normalized_names))
name_to_id = {name: idx for idx, name in enumerate(new_names)}

id_mapping = {
    old_id: name_to_id[normalized_names[old_id]]
    for old_id in range(len(old_names))
}

# Tìm thư mục labels có cấu trúc YOLO
label_files = []
for split in ("train", "valid", "test"):
    split_dirs = [
        p for p in output_dir.rglob(split)
        if p.is_dir()
        and (p / "labels").is_dir()
        and (p / "images").is_dir()
    ]

    for split_dir in split_dirs:
        label_files.extend((split_dir / "labels").rglob("*.txt"))

updated_boxes = 0

for label_file in label_files:
    lines = label_file.read_text(encoding="utf-8").splitlines()
    new_lines = []

    for line in lines:
        parts = line.split()

        if not parts:
            continue

        old_id = int(parts[0])
        if old_id not in id_mapping:
            raise ValueError(
                f"Class ID {old_id} ngoài phạm vi trong {label_file}"
            )

        new_id = id_mapping[old_id]
        if new_id != old_id:
            updated_boxes += 1

        parts[0] = str(new_id)
        new_lines.append(" ".join(parts))

    label_file.write_text(
        "\n".join(new_lines) + ("\n" if new_lines else ""),
        encoding="utf-8"
    )

# Cập nhật data.yaml mới
new_yaml_path = output_dir / data_yaml.relative_to(dataset_dir)

data["nc"] = len(new_names)
data["names"] = new_names

with open(new_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print("=== HOÀN TẤT ===")
print("Dataset gốc:", dataset_dir)
print("Dataset mới:", output_dir)
print("Số class cũ:", len(old_names))
print("Số class mới:", len(new_names))
print("Số file nhãn xử lý:", len(label_files))
print("Số bounding box đổi class ID:", updated_boxes)
print("data.yaml mới:", new_yaml_path)

# 3. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

DATA_YAML = "/content/bep_nha_dataset/data.yaml"

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    workers=2,
    device=0,
    project="/content/bep_nha_runs",
    name="ingredients_v1",
    seed=42
)

In [ ]:
from pathlib import Path

best_model_path = Path(
    "/content/bep_nha_runs/ingredients_v1/weights/best.pt"
)

assert best_model_path.exists(), (
    "Chưa tìm thấy best.pt. Kiểm tra log train."
)

best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=640,
    device=0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
from google.colab import files

uploaded_test = files.upload()
test_image = next(iter(uploaded_test.keys()))

best_model.predict(
    source=test_image,
    conf=0.25,
    save=True
)

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

root = Path("/content/bep_nha_dataset")

yaml_files = list(root.rglob("data.yaml"))
assert yaml_files, "Không tìm thấy data.yaml"

config = yaml.safe_load(yaml_files[0].read_text())
names = config["names"]

if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

# Chỉ đếm ảnh thuộc các thư mục nhãn YOLO
label_dirs = [
    p for p in root.rglob("labels")
    if p.is_dir()
]

box_counts = Counter()
image_counts = Counter()
invalid_lines = []

for label_dir in label_dirs:
    for label_file in label_dir.rglob("*.txt"):
        seen_classes = set()

        for line_no, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):
            parts = line.strip().split()

            if not parts:
                continue

            try:
                class_id = int(parts[0])
            except ValueError:
                invalid_lines.append(
                    (str(label_file), line_no, line)
                )
                continue

            if class_id not in names:
                invalid_lines.append(
                    (str(label_file), line_no, line)
                )
                continue

            box_counts[class_id] += 1
            seen_classes.add(class_id)

        for class_id in seen_classes:
            image_counts[class_id] += 1

print("=== Thống kê class ===")
for class_id, name in names.items():
    print(
        f"{name}: "
        f"{image_counts[class_id]} ảnh, "
        f"{box_counts[class_id]} bounding boxes"
    )

print(f"\nSố dòng nhãn không hợp lệ: {len(invalid_lines)}")
for item in invalid_lines[:10]:
    print(item)

In [ ]:
from pathlib import Path

for p in Path("/content").rglob("best.pt"):
    print(p)

In [ ]:
# các class đã học được
from ultralytics import YOLO

MODEL_PATH = "/content/bep_nha_runs/ingredients_v1/weights/best.pt"

model = YOLO(MODEL_PATH)

print("Các class của model:")
print(model.names)

In [ ]:
# kiểm tra validation của từng class

from pathlib import Path

DATA_YAML = "/content/bep_nha_dataset/data.yaml"

# Tìm đúng data.yaml nếu nằm trong thư mục con
if not Path(DATA_YAML).exists():
    files = list(Path("/content/bep_nha_dataset").rglob("data.yaml"))
    print("Các file data.yaml tìm được:", files)
    DATA_YAML = str(files[0])

metrics = model.val(
    data=DATA_YAML,
    imgsz=640,
    device=0
)

print("Tên class:", metrics.names)
print("Precision từng class:", metrics.box.p)
print("Recall từng class:", metrics.box.r)
print("mAP50 từng class:", metrics.box.maps)

In [ ]:
print("Số class:", len(metrics.names))
print("Số precision:", len(metrics.box.p))
print("Số recall:", len(metrics.box.r))
print("Số mAP50:", len(metrics.box.maps))

for class_id, name in metrics.names.items():
    print(
        class_id,
        name,
        "P =", metrics.box.p[class_id]
        if class_id < len(metrics.box.p) else "N/A",
        "R =", metrics.box.r[class_id]
        if class_id < len(metrics.box.r) else "N/A",
        "mAP50 =", metrics.box.maps[class_id]
        if class_id < len(metrics.box.maps) else "N/A",
    )

# 4. Check

In [ ]:

from ultralytics import YOLO
from IPython.display import display
from PIL import Image
from pathlib import Path

# 1. Load model đã train
model = YOLO(
    "/content/bep_nha_runs/ingredients_v1/weights/best.pt"
)

# 2. Đường dẫn ảnh cần kiểm tra
image_paths = [
    "/content/thit-bo.jpg",
    "/content/thit-lon.jpeg",
    "/content/trung-ga.jpeg",
    "/content/hanh-la.jpeg",
    "/content/hanh-tay.jpg",
]

# 3. Kiểm tra file ảnh tồn tại
for path in image_paths:
    if not Path(path).is_file():
        raise FileNotFoundError(f"Không tìm thấy ảnh: {path}")

# 4. Chạy nhận diện
results = model.predict(
    source=image_paths,
    imgsz=640,
    conf=0.1,
    save=True,
    project="/content/bep_nha_test",
    name="meat",
    exist_ok=True,
)

# 5. Hiển thị kết quả
for result in results:
    print(f"\nẢnh: {Path(result.path).name}")

    if len(result.boxes) == 0:
        print("Không phát hiện đối tượng nào.")
    else:
        for box in result.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())

            print(
                f"{model.names[class_id]}: "
                f"{confidence:.3f}"
            )

    # Hiển thị ảnh có bounding box trực tiếp từ YOLO
    plotted_image = result.plot()
    display(Image.fromarray(plotted_image[:, :, ::-1]))

# 6. In đường dẫn thư mục đã lưu ảnh kết quả
print("\nẢnh kết quả đã lưu tại:")
print("/content/bep_nha_test/meat/")


In [ ]:
from IPython.display import Image, display

test_image = "hanh-cu.jpg"

display(Image(filename="/content/runs/detect/predict/" + test_image))

# Tải model

In [ ]:
from google.colab import files

best_model_path = "/content/bep_nha_runs/ingredients_v2/weights/best.pt"

files.download(str(best_model_path))

# So sánh hai version

In [ ]:
from ultralytics import YOLO

DATA_YAML = "/content/bep_nha_dataset/data.yaml"

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=8,
    patience=15,
    workers=2,
    device=0,

    optimizer="AdamW",
    lr0=0.001,

    seed=42,
    deterministic=True,

    project="/content/bep_nha_runs",
    name="ingredients_v2",
    exist_ok=False,
    plots=True,
    save=True
)

print("Kết quả lưu tại:", results.save_dir)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = "/content/bep_nha_dataset/data.yaml"

V1_PATH = Path(
    "/content/bep_nha_runs/ingredients_v1/weights/best.pt"
)

V2_PATH = Path(
    "/content/bep_nha_runs/ingredients_v2/weights/best.pt"
)

assert V1_PATH.is_file(), f"Không tìm thấy {V1_PATH}"
assert V2_PATH.is_file(), f"Không tìm thấy {V2_PATH}"

v1 = YOLO(str(V1_PATH))
v2 = YOLO(str(V2_PATH))

m1 = v1.val(data=DATA_YAML, split="val")
m2 = v2.val(data=DATA_YAML, split="val")

print(f"{'Metric':<15}{'V1':>12}{'V2':>12}")
print("-" * 39)

for name, a, b in [
    ("Precision", m1.box.mp, m2.box.mp),
    ("Recall", m1.box.mr, m2.box.mr),
    ("mAP50", m1.box.map50, m2.box.map50),
    ("mAP50-95", m1.box.map, m2.box.map),
]:
    print(f"{name:<15}{a:>12.4f}{b:>12.4f}")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image

# Đường dẫn model V2
MODEL_PATH = Path(
    "/content/bep_nha_runs/ingredients_v2/weights/best.pt"
)

# Kiểm tra file model
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy model V2: {MODEL_PATH}\n"
        "Hãy kiểm tra đường dẫn file best.pt của lần train V2."
    )

model = YOLO(str(MODEL_PATH))

print("Đã load model V2")
print("Số class:", len(model.names))
print("Danh sách class:", model.names)

image_paths = [
    "/content/ca-chua-1.jpeg",
    "/content/ca-rot-2.jpeg",
    "/content/ca-rot-3.jpeg",
    "/content/hanh-la-2.jpeg",
]

# Chỉ giữ lại các ảnh tồn tại
image_paths = [p for p in image_paths if Path(p).exists()]

if not image_paths:
    raise FileNotFoundError(
        "Không tìm thấy ảnh nào. Hãy sửa image_paths thành đường dẫn ảnh đã upload."
    )

results = model.predict(
    source=image_paths,
    imgsz=640,
    conf=0.15,
    iou=0.45,
    save=True,
    project="/content/bep_nha_predictions",
    name="v2",
    exist_ok=True,
    verbose=False
)

# Hiển thị ảnh đã vẽ bounding box
from IPython.display import display
from PIL import Image as PILImage
from pathlib import Path

for result in results:
    print(f"\nẢnh: {Path(result.path).name}")

    if result.boxes is None or len(result.boxes) == 0:
        print("Không phát hiện được nguyên liệu.")
    else:
        for box in result.boxes:
            class_id = int(box.cls[0].item())
            confidence = float(box.conf[0].item())
            class_name = model.names[class_id]

            print(f"- {class_name}: {confidence:.3f}")

    # result.plot() trả về mảng ảnh BGR
    plotted_image = result.plot()

    # Chuyển BGR sang RGB rồi hiển thị
    display(PILImage.fromarray(plotted_image[:, :, ::-1]))